# Phase 3A: Data Understanding, Real Schema Inspection & Leakage Audit

**Project Title**: AI-Based Sales Conversion Probability Prediction  
**Dataset**: UCI Online Shoppers Purchasing Intention Dataset (Sakar et al., 2018)  
**Source**: `data/raw/online_shoppers_intention.csv`  

---  
## System & Prediction Definitions
- **Prediction Unit**: An individual online e-commerce browsing session.
- **Prediction Target**: `Revenue` (Boolean: `True` if session ends in a purchase transaction, `False` otherwise).
- **Prediction Timing**: Real-time during session navigation prior to final checkout submission.
- **Available at Prediction Time**: Session page counts, browsing durations, Exit/Bounce rates, historical page value lookup metrics (`PageValues`), and visitor context (Month, VisitorType, Weekend, OS, Browser, Region, TrafficType).
- **Unavailable / Target Leakage**: Post-transaction payment flags, deal closure notes, or post-purchase timestamps (None present in this dataset schema).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/raw/online_shoppers_intention.csv")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Schema & Data Types Inspection

In [ ]:
schema_df = pd.DataFrame({
    "Data_Type": df.dtypes,
    "Missing_Count": df.isnull().sum(),
    "Missing_Pct": (df.isnull().sum() / len(df)) * 100,
    "Unique_Values": df.nunique()
})
print("=== SCHEMA AUDIT ===")
display(schema_df)
print(f"Total Duplicate Rows: {df.duplicated().sum()} ({(df.duplicated().sum()/len(df))*100:.2f}%)")

## 2. Target Class Distribution (`Revenue`)

In [ ]:
target_counts = df["Revenue"].value_counts(dropna=False)
target_pcts = df["Revenue"].value_counts(normalize=True, dropna=False) * 100
target_summary = pd.DataFrame({"Count": target_counts, "Percentage": target_pcts})
print("=== TARGET VARIABLE DISTRIBUTION ===")
display(target_summary)

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Revenue", palette="Set2")
plt.title("Target Class Distribution (Revenue)", fontweight="bold")
plt.show()

## 3. Numerical & Categorical Summaries

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
print("=== NUMERICAL DESCRIPTIVE STATISTICS ===")
display(df[num_cols].describe().T)

cat_cols = df.select_dtypes(include=["object", "bool"]).columns
print("\n=== CATEGORICAL FREQUENCY DISTRIBUTIONS ===")
for c in cat_cols:
    if c != "Revenue":
        print(f"\n--- {c} ---")
        display(df[c].value_counts(dropna=False))

## 4. Leakage Audit & Preliminary Feature Eligibility Table

Every feature in `online_shoppers_intention.csv` has been evaluated against prediction timing and business availability:

| Feature | Meaning | Data Type | Available at Prediction Time? | Leakage Risk | Decision | Reason |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| `Administrative` | Admin pages visited | int64 | Yes | Low | Safe | Session navigation count |
| `Administrative_Duration` | Admin page time (s) | float64 | Yes | Low | Safe | Session duration metric |
| `Informational` | Info pages visited | int64 | Yes | Low | Safe | Session navigation count |
| `Informational_Duration` | Info page time (s) | float64 | Yes | Low | Safe | Session duration metric |
| `ProductRelated` | Product pages visited | int64 | Yes | Low | Safe | Core browsing engagement |
| `ProductRelated_Duration` | Product page time (s) | float64 | Yes | Low | Safe | Core browsing duration |
| `BounceRates` | Avg page bounce rate | float64 | Yes | Low | Safe | Historical web tracking metric |
| `ExitRates` | Avg page exit rate | float64 | Yes | Low | Safe | Historical web tracking metric |
| `PageValues` | Avg Google Analytics page value | float64 | Yes | Medium (Risky) | **Risky Candidate (Monitor)** | Historical page lookup value. Available at prediction time, but highly predictive (mean=27.26 for True vs 1.98 for False). |
| `SpecialDay` | Closeness to holiday | float64 | Yes | Low | Safe | Calendar attribute |
| `Month` | Visit month | object | Yes | Low | Safe | Seasonality attribute |
| `OperatingSystems` | OS ID | int64 | Yes | Low | Safe | Tech environment attribute |
| `Browser` | Browser ID | int64 | Yes | Low | Safe | Tech environment attribute |
| `Region` | Geographic region ID | int64 | Yes | Low | Safe | Geolocation attribute |
| `TrafficType` | Traffic source type ID | int64 | Yes | Low | Safe | Acquisition channel attribute |
| `VisitorType` | Visitor type | object | Yes | Low | Safe | User history attribute |
| `Weekend` | Weekend boolean | bool | Yes | Low | Safe | Calendar attribute |